# GeroQuery — end-to-end example

Query a gene, apply an aging clock, and compute a resilience signal — all through the same service the REST API exposes.

In [ ]:
from geroquery.api.service import GeroService
svc = GeroService()
svc.version()

## 1. Gene query — cross-species aging signature

In [ ]:
import pandas as pd
card = svc.gene_card('CDKN2A')  # p16INK4a
print(card['gene']['symbol'], card['gene']['canonical_id'])
pd.DataFrame(card['meta_signatures'])[['omic_layer','species','pooled_effect','ci_low','ci_high','n_studies','direction']]

In [ ]:
# Curated knowledge + linked interventions
print('curated:', [(f['database'], f['assertion']) for f in card['curated_flags']])
print('interventions:', [i['name'] for i in card['interventions']])

## 2. Aging clock — predicted age & acceleration

In [ ]:
df = svc.store.get_dataset('clinical_nhanes_slice')
res = svc.apply_clock('clinical_phenoage_demo', df, df['age'].tolist())
print('predicts:', res['predicted_outcome'], '| mean age acceleration:', round(res['mean_age_acceleration'], 3))

## 3. Resilience — critical slowing down with age

In [ ]:
csd = svc.resilience_csd(dataset_id='clinical_nhanes_slice', n_strata=6)
print('resilience declines with age:', csd['resilience_declines'])
print('method:', csd['method'])
pd.DataFrame({'age': csd['strata_midpoints'], 'variance': csd['variance'], 'cross_correlation': csd['cross_correlation']})